# Chat with Your Own Data

## How It Works

1. **PDF Upload & Text Extraction**  
   - The user uploads a PDF through the Panel interface.  
   - The app reads the file using `pypdf` and extracts all text from its pages.  
   - If the PDF is image-based (scanned), no text will be extracted unless OCR (e.g., `pytesseract`) is added.

2. **Chunking the Document**  
   - The extracted text is split into overlapping chunks (e.g., 1,000 words per chunk with 150-word overlap).  
   - Overlapping ensures context continuity between adjacent chunks.  
   - Each chunk becomes a self-contained passage for retrieval.

3. **Embedding the Chunks**  
   - Each chunk is converted into a numerical embedding vector using the **OpenAI Embeddings API** (`text-embedding-3-small`).  
   - These embeddings capture semantic meaning and allow similarity comparison between the user’s query and document text.

4. **Question Processing**  
   - When a user enters a question, it is also embedded using the same model.  
   - The system computes cosine similarity between the question vector and all document chunk vectors.  
   - The **top K most relevant chunks** (default: 5) are selected as the context for answering.

5. **Answer Generation (Grounded Chat)**  
   - The selected chunks are concatenated into a structured context and sent to an OpenAI chat model (e.g., `gpt-4o-mini`).  
   - The system message instructs the model to:
     - Answer **only** using the provided excerpts.
     - Politely decline if the answer is not supported by the excerpts.
     - Optionally cite excerpt numbers when helpful.
   - The model generates a grounded, text-based response.

6. **Interactive Display**  
   - The conversation (user question + model answer) is displayed in the Panel app dynamically.  
   - The user can continue asking more questions, all based on the uploaded PDF, or quit the app.

**Result:**  
You get a lightweight local Retrieval-Augmented Generation (RAG) app that answers questions strictly grounded in your uploaded PDF — no external data, no hallucination.


In [ ]:
pip install pypdf

In [1]:
# --- Imports & setup ---
import os, io, numpy as np
from dotenv import load_dotenv, find_dotenv
import panel as pn
from pypdf import PdfReader
import openai

# define the environment path. it is at the parent folder

env_path ="../.env"
load_dotenv(dotenv_path=env_path, override=True)

pn.extension()

# --- Globals for the in-memory "index" ---
doc_text = ""
doc_chunks = []
doc_embs = None  # numpy array shape (n_chunks, dim)

# --- Parameters you can tweak ---
CHUNK_WORDS = 1000       # ~1000 words per chunk
CHUNK_OVERLAP = 150      # words of overlap
TOP_K = 5                # how many chunks to retrieve for each question
EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"


# ---------- Utilities ----------
def extract_text_from_pdf_bytes(pdf_bytes: bytes) -> str:
    reader = PdfReader(io.BytesIO(pdf_bytes))
    pages = []
    for p in reader.pages:
        pages.append(p.extract_text() or "")
    return "\n".join(pages)


def chunk_text_words(text: str, chunk_words=CHUNK_WORDS, overlap_words=CHUNK_OVERLAP):
    words = text.split()
    if not words:
        return []
    step = max(1, chunk_words - overlap_words)
    chunks = []
    for start in range(0, len(words), step):
        chunk = " ".join(words[start : start + chunk_words])
        if chunk.strip():
            chunks.append(chunk)
        if start + chunk_words >= len(words):
            break
    return chunks


def embed_texts(texts):
    resp = openai.embeddings.create(model=EMBED_MODEL, input=texts)
    vecs = [d.embedding for d in resp.data]
    return np.array(vecs, dtype="float32")


def cosine_sim_matrix(a, b):
    a_norm = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)
    b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-8)
    return a_norm @ b_norm.T


def retrieve_context(question: str, top_k=TOP_K):
    if not doc_chunks:
        return ""
    q_emb = embed_texts([question])  # (1, d)
    sims = cosine_sim_matrix(q_emb, doc_embs).ravel()  # (n_chunks,)
    top_idx = np.argsort(sims)[-top_k:][::-1]
    selected = [doc_chunks[i] for i in top_idx]
    return "\n\n".join(f"[Excerpt {i+1}]\n{txt}" for i, txt in enumerate(selected))


def ask_grounded(question: str, context: str, temperature=0):
    system_msg = (
        "You are a careful research assistant. "
        "Answer ONLY using the provided excerpts. "
        "If the answer is not contained in the excerpts, say you don't have enough information. "
        "Cite the excerpt numbers when helpful."
    )
    user_msg = f"Question:\n{question}\n\nExcerpts:\n{context}"

    resp = openai.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
        temperature=temperature,
    )
    return resp.choices[0].message.content


# ---------- Panel UI ----------
inp = pn.widgets.TextInput(placeholder="Ask a question about the uploaded PDF…", width=600)
ask_btn = pn.widgets.Button(name="Ask", button_type="primary")
quit_btn = pn.widgets.Button(name="Quit", button_type="warning")
clear_btn = pn.widgets.Button(name="Clear History", button_type="default")
uploader = pn.widgets.FileInput(accept=".pdf", multiple=False)
status = pn.pane.Markdown("**No PDF uploaded.**")

# Chat area (newest on TOP): we'll PREPEND each exchange at index 0
chat_area = pn.Column(sizing_mode="stretch_width", height=600, scroll=True)

def prepend_exchange(user_text: str, assistant_text: str):
    """Create a single 'exchange' panel and insert at the top (index 0)."""
    exchange = pn.Column(
        pn.Row("**User:**", pn.pane.Markdown(user_text, width=800)),
        pn.Row("**Assistant:**", pn.pane.Markdown(assistant_text, width=800)),
        pn.layout.Divider(),
        sizing_mode="stretch_width",
    )
    chat_area.insert(0, exchange)  # <-- newest on top

def prepend_notice(notice_md: str):
    notice = pn.Column(
        pn.pane.Markdown(notice_md, width=800),
        pn.layout.Divider(),
        sizing_mode="stretch_width",
    )
    chat_area.insert(0, notice)


def on_upload(event):
    global doc_text, doc_chunks, doc_embs
    if not uploader.value:
        status.object = "**No PDF uploaded.**"
        return

    try:
        status.object = "Processing PDF… (extracting text)"
        pdf_bytes = uploader.value
        doc_text = extract_text_from_pdf_bytes(pdf_bytes)

        if not doc_text.strip():
            status.object = ":warning: Could not extract text from this PDF."
            return

        status.object = "Chunking & embedding…"
        doc_chunks = chunk_text_words(doc_text, CHUNK_WORDS, CHUNK_OVERLAP)
        doc_embs = embed_texts(doc_chunks)

        status.object = (
            f"✅ PDF loaded. Built {len(doc_chunks)} chunks for retrieval."
        )
        prepend_notice("**PDF loaded and indexed.** You can start asking questions.")
    except Exception as e:
        status.object = f"Error processing PDF  `{e}`"

uploader.param.watch(on_upload, "value")


def on_ask(event):
    question = inp.value.strip()
    if not question:
        return
    if not doc_chunks:
        prepend_exchange(question, "Please upload a PDF first; I can only answer questions based on the uploaded document.")
        return

    context = retrieve_context(question, TOP_K)
    answer = ask_grounded(question, context)
    prepend_exchange(question, answer)
    inp.value = ""


def on_clear(event):
    chat_area.clear()
    prepend_notice("_Conversation history cleared._")


def on_quit(event):
    chat_area.clear()
    status.object = "Application closed."
    try:
        pn.io.server.stop()
    except Exception:
        pass


ask_btn.on_click(on_ask)
clear_btn.on_click(on_clear)
quit_btn.on_click(on_quit)

dashboard = pn.Column(
    pn.pane.Markdown("# Chat with Your PDF"),
    pn.Row(pn.Column("**Upload a PDF:**", uploader, status, width=450)),
    pn.Spacer(height=5),
    pn.Row(inp, ask_btn, quit_btn, clear_btn),
    pn.layout.Divider(),
    chat_area,
    sizing_mode="stretch_width",
)

dashboard

Column(sizing_mode='stretch_width')
    [0] Markdown(str)
    [1] Row
        [0] Column(width=450)
            [0] Markdown(str)
            [1] FileInput(accept='.pdf')
            [2] Markdown(str)
    [2] Spacer(height=5)
    [3] Row
        [0] TextInput(placeholder='Ask a question a..., width=600)
        [1] Button(button_type='primary', name='Ask')
        [2] Button(button_type='warning', name='Quit')
        [3] Button(name='Clear History')
    [4] Divider()
    [5] Column(height=600, scroll=True, sizing_mode='stretch_width')

## Try experimenting on your own!

You can modify the menu or instructions to create your own orderbot!